# Module 3b: Ray Data - Distributed Datasets

**DSC 232R - Big Data Analysis Using Spark**

This notebook covers Ray Data:
1. Creating and loading datasets
2. Transformations (map, filter, flat_map)
3. Batch operations for ML
4. Integration with pandas and numpy

**Prerequisites**: `pip install "ray[data]"`

## Key Takeaways

- **Ray Data** provides distributed datasets optimized for ML pipelines
- **Lazy evaluation** like Spark - transformations build a plan
- **`map_batches()`** is the key method for efficient ML preprocessing
- **Seamless integration** with pandas, numpy, and Ray Train

In [ ]:
import ray
import numpy as np
import pandas as pd
import time

# Initialize Ray
if ray.is_initialized():
    ray.shutdown()

ray.init(num_cpus=4, logging_level="WARNING")
print(f"Ray version: {ray.__version__}")

---

## 1. Creating Datasets

### From Python Collections

In [ ]:
# From a list of dictionaries
data = [
    {"name": "Alice", "age": 25, "city": "NYC"},
    {"name": "Bob", "age": 30, "city": "LA"},
    {"name": "Charlie", "age": 35, "city": "NYC"},
    {"name": "Diana", "age": 28, "city": "Chicago"},
]

ds = ray.data.from_items(data)
print(f"Dataset: {ds}")
print(f"Count: {ds.count()}")
print(f"\nSchema: {ds.schema()}")

In [ ]:
# View first few rows
ds.take(3)

### From Pandas DataFrame

In [ ]:
# Create pandas DataFrame
pdf = pd.DataFrame({
    "temperature": np.random.uniform(20, 35, 1000),
    "humidity": np.random.uniform(30, 90, 1000),
    "pressure": np.random.uniform(990, 1030, 1000),
    "wind_speed": np.random.uniform(0, 30, 1000),
})

# Convert to Ray Dataset
ds_weather = ray.data.from_pandas(pdf)
print(f"Dataset: {ds_weather}")
print(f"\nFirst 3 rows:")
ds_weather.take(3)

### From NumPy Arrays

In [ ]:
# Create numpy array
np_data = np.random.rand(1000, 5)

# Convert to Ray Dataset
ds_numpy = ray.data.from_numpy(np_data)
print(f"Dataset: {ds_numpy}")
print(f"\nFirst 3 rows:")
ds_numpy.take(3)

### From Files (Parquet, CSV, JSON)

In [ ]:
# Save sample data for demonstration
import os
import tempfile

temp_dir = tempfile.mkdtemp()

# Create sample parquet files
for i in range(3):
    chunk = pd.DataFrame({
        "id": range(i*100, (i+1)*100),
        "value": np.random.randn(100),
        "category": np.random.choice(["A", "B", "C"], 100)
    })
    chunk.to_parquet(f"{temp_dir}/data_{i}.parquet")

print(f"Created files in {temp_dir}")
print(os.listdir(temp_dir))

In [ ]:
# Read parquet files (can use glob patterns)
ds_parquet = ray.data.read_parquet(f"{temp_dir}/*.parquet")
print(f"Dataset: {ds_parquet}")
print(f"\nFirst 3 rows:")
ds_parquet.take(3)

---

## 2. Basic Transformations

### map() - Transform Each Row

In [ ]:
# Create dataset
ds = ray.data.from_items([{"x": i} for i in range(10)])

# Map: apply function to each row
ds_mapped = ds.map(lambda row: {"x": row["x"], "x_squared": row["x"] ** 2})

print("After map:")
ds_mapped.take(5)

### filter() - Select Rows

In [ ]:
# Filter: keep rows matching condition
ds_filtered = ds_mapped.filter(lambda row: row["x"] > 5)

print("After filter (x > 5):")
ds_filtered.take_all()

### flat_map() - One to Many

In [ ]:
# Flat map: one row becomes multiple rows
ds_small = ray.data.from_items([{"nums": [1, 2, 3]}, {"nums": [4, 5]}])

ds_flat = ds_small.flat_map(lambda row: [{"num": n} for n in row["nums"]])

print("Original:")
print(ds_small.take_all())
print("\nAfter flat_map:")
print(ds_flat.take_all())

---

## 3. Batch Operations (map_batches)

**`map_batches()`** is the most important method for ML preprocessing. It processes multiple rows at once, enabling vectorized operations.

In [ ]:
# Create dataset with numeric columns
ds_numeric = ray.data.from_pandas(pd.DataFrame({
    "feature1": np.random.randn(1000),
    "feature2": np.random.randn(1000) * 2 + 5,
    "feature3": np.random.randn(1000) * 0.5 - 1,
}))

print(f"Dataset: {ds_numeric}")

In [ ]:
# Define batch processing function
def normalize_batch(batch: pd.DataFrame) -> pd.DataFrame:
    """Normalize all columns to [0, 1] range within batch."""
    result = batch.copy()
    for col in batch.columns:
        min_val = batch[col].min()
        max_val = batch[col].max()
        if max_val > min_val:
            result[col] = (batch[col] - min_val) / (max_val - min_val)
    return result

# Apply to batches
ds_normalized = ds_numeric.map_batches(normalize_batch, batch_format="pandas")

print("Normalized sample:")
ds_normalized.take(3)

### Batch Processing with NumPy

In [ ]:
# Process as numpy arrays (efficient for numeric operations)
def compute_features_numpy(batch: dict) -> dict:
    """Compute additional features from numpy batch."""
    f1 = batch["feature1"]
    f2 = batch["feature2"]
    f3 = batch["feature3"]
    
    return {
        "feature1": f1,
        "feature2": f2,
        "feature3": f3,
        "f1_squared": f1 ** 2,
        "f1_f2_product": f1 * f2,
        "feature_sum": f1 + f2 + f3,
    }

ds_features = ds_numeric.map_batches(compute_features_numpy, batch_format="numpy")

print("With additional features:")
ds_features.take(3)

### Controlling Batch Size

In [ ]:
def print_batch_info(batch: pd.DataFrame) -> pd.DataFrame:
    """Print info about each batch."""
    print(f"  Processing batch with {len(batch)} rows")
    return batch

# Default batch size
print("Default batch size:")
_ = ds_numeric.map_batches(print_batch_info, batch_format="pandas").take(1)

# Custom batch size
print("\nBatch size = 100:")
_ = ds_numeric.map_batches(
    print_batch_info, 
    batch_format="pandas",
    batch_size=100
).take(1)

---

## 4. Aggregations

In [ ]:
# Create dataset with groups
ds_grouped = ray.data.from_items([
    {"city": "NYC", "temperature": 75, "humidity": 60},
    {"city": "NYC", "temperature": 78, "humidity": 55},
    {"city": "LA", "temperature": 85, "humidity": 40},
    {"city": "LA", "temperature": 82, "humidity": 45},
    {"city": "Chicago", "temperature": 68, "humidity": 70},
    {"city": "Chicago", "temperature": 65, "humidity": 75},
])

# Global aggregations
print("Global stats:")
print(f"  Min temperature: {ds_grouped.min('temperature')}")
print(f"  Max temperature: {ds_grouped.max('temperature')}")
print(f"  Mean temperature: {ds_grouped.mean('temperature')}")

In [ ]:
# Group by aggregations
ds_by_city = ds_grouped.groupby("city").mean(["temperature", "humidity"])

print("Grouped by city:")
ds_by_city.take_all()

---

## 5. Train/Test Split

In [ ]:
# Create a larger dataset
ds_full = ray.data.from_pandas(pd.DataFrame({
    "feature": np.random.randn(1000),
    "label": np.random.randint(0, 2, 1000)
}))

# Split into train/test
train_ds, test_ds = ds_full.train_test_split(test_size=0.2, shuffle=True)

print(f"Full dataset: {ds_full.count()} rows")
print(f"Train dataset: {train_ds.count()} rows")
print(f"Test dataset: {test_ds.count()} rows")

---

## 6. Converting Back to Pandas/NumPy

In [ ]:
# To pandas
pdf_result = train_ds.to_pandas()
print(f"Pandas DataFrame shape: {pdf_result.shape}")
print(pdf_result.head())

In [ ]:
# Iterate in batches (memory efficient for large datasets)
print("Iterating in batches:")
for i, batch in enumerate(train_ds.iter_batches(batch_size=200, batch_format="pandas")):
    print(f"  Batch {i}: {len(batch)} rows")
    if i >= 2:
        print("  ...")
        break

---

## 7. Exercise: Weather Data Pipeline

Create a data processing pipeline for weather data:

In [ ]:
# Generate sample weather data
np.random.seed(42)
n_samples = 5000

weather_data = pd.DataFrame({
    "station_id": np.random.choice(["A", "B", "C", "D"], n_samples),
    "temperature_kelvin": np.random.uniform(260, 310, n_samples),  # Kelvin
    "humidity_pct": np.random.uniform(20, 100, n_samples),
    "pressure_hpa": np.random.uniform(980, 1040, n_samples),
    "wind_speed_ms": np.random.uniform(0, 25, n_samples),
})

ds_weather = ray.data.from_pandas(weather_data)
print(f"Weather dataset: {ds_weather}")

In [ ]:
# Exercise: Complete the pipeline

def convert_temperature(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Convert temperature from Kelvin to Celsius.
    Celsius = Kelvin - 273.15
    """
    # Your code here
    pass

def compute_heat_index(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Compute a simplified heat index.
    heat_index = temperature_celsius + (humidity_pct * 0.1)
    """
    # Your code here
    pass

def categorize_wind(batch: pd.DataFrame) -> pd.DataFrame:
    """
    Add wind category:
    - calm: < 5 m/s
    - moderate: 5-15 m/s
    - strong: > 15 m/s
    """
    # Your code here
    pass

# Build pipeline
# ds_processed = ds_weather \
#     .map_batches(convert_temperature, batch_format="pandas") \
#     .map_batches(compute_heat_index, batch_format="pandas") \
#     .map_batches(categorize_wind, batch_format="pandas")

# print("Processed sample:")
# ds_processed.take(5)

In [ ]:
# Solution

def convert_temperature_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    result["temperature_celsius"] = result["temperature_kelvin"] - 273.15
    return result

def compute_heat_index_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    result["heat_index"] = result["temperature_celsius"] + (result["humidity_pct"] * 0.1)
    return result

def categorize_wind_solution(batch: pd.DataFrame) -> pd.DataFrame:
    result = batch.copy()
    conditions = [
        result["wind_speed_ms"] < 5,
        (result["wind_speed_ms"] >= 5) & (result["wind_speed_ms"] <= 15),
        result["wind_speed_ms"] > 15
    ]
    choices = ["calm", "moderate", "strong"]
    result["wind_category"] = np.select(conditions, choices)
    return result

# Build pipeline
ds_processed = ds_weather \
    .map_batches(convert_temperature_solution, batch_format="pandas") \
    .map_batches(compute_heat_index_solution, batch_format="pandas") \
    .map_batches(categorize_wind_solution, batch_format="pandas")

print("Processed sample:")
for row in ds_processed.take(5):
    print(f"  Station {row['station_id']}: {row['temperature_celsius']:.1f}C, "
          f"Heat Index: {row['heat_index']:.1f}, Wind: {row['wind_category']}")

In [ ]:
# Analyze by station
print("\nStatistics by station:")
station_stats = ds_processed.groupby("station_id").mean(["temperature_celsius", "heat_index"])

for row in station_stats.take_all():
    print(f"  Station {row['station_id']}: "
          f"Avg Temp: {row['mean(temperature_celsius)']:.1f}C, "
          f"Avg Heat Index: {row['mean(heat_index)']:.1f}")

---

## 8. Ray Data vs Spark Comparison

In [ ]:
comparison = pd.DataFrame({
    "Feature": [
        "Primary use case",
        "Data model",
        "Execution",
        "SQL support",
        "ML integration",
        "Best for"
    ],
    "Spark": [
        "ETL, data warehousing",
        "DataFrame (tabular)",
        "JVM-based",
        "Excellent (Spark SQL)",
        "MLlib (limited)",
        "Joins, aggregations, SQL"
    ],
    "Ray Data": [
        "ML preprocessing",
        "Dataset (row-based)",
        "Native Python",
        "Limited",
        "Excellent (Ray Train)",
        "Feature engineering, batching"
    ]
})

print("Ray Data vs Spark DataFrame")
print("=" * 70)
print(comparison.to_string(index=False))

---

## Summary

### Key Methods

| Method | Purpose | Example |
|--------|---------|--------|
| `map()` | Transform each row | `ds.map(lambda r: {...})` |
| `filter()` | Select rows | `ds.filter(lambda r: r["x"] > 0)` |
| `flat_map()` | One to many | `ds.flat_map(lambda r: [...])` |
| `map_batches()` | Vectorized ops | `ds.map_batches(func)` |
| `groupby()` | Aggregations | `ds.groupby("col").mean()` |

### Best Practices

1. Use `map_batches()` for numeric operations (vectorized)
2. Use `batch_format="pandas"` or `"numpy"` based on your operation
3. For ML: use `train_test_split()` and `iter_batches()`
4. Use Spark for complex joins/SQL; Ray Data for ML preprocessing

### Next: Ray Train

See `03c_ray_train.ipynb` for distributed model training.

In [ ]:
# Cleanup
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)
ray.shutdown()
print("Cleanup complete.")